In [1]:
pip install -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 MB 14.6 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 15.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.5/19.5 MB 16.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 16.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 13.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 15.2 MB/s eta 0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.2.3
    Uninstalling urllib3-2.2.3:
      Successfully uninstalled urllib3-2.2.3
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0
    Uninstalling python-dateutil-2.9.0:
      Successfully uninstalled python-dateutil-2.9.0
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled n

In [29]:
import os
import json
import boto3
import glob
from urllib.parse import unquote_plus
from pyspark.sql import SparkSession
import uuid
import sys
import psycopg2
import logging
import pandas as pd
import janitor
from datetime import datetime
from pandas.api.types import infer_dtype
from pyspark.sql.functions import col, trim, regexp_replace, row_number, sha2, concat_ws, conv, lit, concat, coalesce, isnan,  isnull, regexp_replace,  monotonically_increasing_id, format_string
import tempfile
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, DoubleType, StringType, TimestampType, LongType
import numpy as np
import hashlib
from pyspark.sql.functions import udf
import warnings
from typing import Dict, Optional

In [2]:
ENDPOINT=os.getenv('ENDPOINT')
PORT=int(os.getenv('PORT'))
USER=os.getenv('USER')
REGION=os.getenv('REGION')
DBNAME=os.getenv('DBNAME')
DESTINATION_BUCKETNAME=os.getenv('DESTINATION_BUCKETNAME')

In [3]:
def setup_logger(name: str = None) -> logging.Logger:
    logger = logging.getLogger(name)
    if not logger.handlers:
        formatter = logging.Formatter('[%(asctime)s] %(levelname)s @ line %(lineno)d: %(message)s')
        handler = logging.StreamHandler(sys.stdout)
        handler.setLevel(logging.INFO)
        handler.setFormatter(formatter)
        logger.addHandler(handler)
        logger.setLevel(logging.INFO)
    return logger

logger = setup_logger(__name__)

In [4]:
client = boto3.client('rds')
s3_client = boto3.client('s3')

In [5]:
os.environ['DBNAME'] = 'piping'
os.environ['DB_PASSWORD']='Asimo1721.'
os.environ['AWS_REGION'] = 'eu-north-1'
def check_environment():
    required_vars = ['ENDPOINT', 'PORT', 'USER', 'REGION', 'DBNAME']
    missing_vars = [var for var in required_vars if not os.getenv(var)]
    if missing_vars:
        raise ValueError(f"Missing required environment variables: {', '.join(missing_vars)}")

In [6]:
check_environment()

In [7]:
dt_string = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")
AppName = "RDSJOB"

In [8]:
current_directory = os.getcwd()
print(f"Current directory: {current_directory}")

Current directory: /home/jovyan


In [10]:
def create_spark_session():
    try:
        jar_path = "/home/jovyan/.jars/postgresql-42.7.5.jar"
        if not os.path.exists(jar_path):
            logger.error(f"PostgreSQL JDBC driver not found at {jar_path}")
            raise FileNotFoundError(f"PostgreSQL JDBC driver not found at {jar_path}")
        
        logger.info(f"PostgreSQL JDBC driver found at {jar_path}")

        spark = SparkSession.builder \
            .appName("AppName") \
            .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
            .config("spark.hadoop.fs.s3a.aws.credentials.provider",
                    "com.amazonaws.auth.DefaultAWSCredentialsProviderChain") \
            .config("spark.driver.extraClassPath", jar_path) \
            .config("spark.executor.extraClassPath", jar_path) \
            .master("local[*]") \
            .getOrCreate()

        return spark

    except Exception as e:
        logger.error(f"Error creating Spark session: {str(e)}")
        raise

In [11]:
sparkjum=create_spark_session()

[2025-04-24 16:36:33,479] INFO @ line 8: PostgreSQL JDBC driver found at /home/jovyan/.jars/postgresql-42.7.5.jar


In [14]:
input_path='TEIGA_TMI_ADISSEO_CONTROL_AVANCE_SOPORTACION.csv'
output_path='output/'

In [15]:
def get_db_connection_params() -> Dict[str, str]:
    """Get database connection parameters from environment variables."""
    required_params = ['ENDPOINT', 'PORT', 'DBNAME', 'USER', 'DB_PASSWORD']
    
    params = {}
    for param in required_params:
        value = os.environ.get(param)
        if not value:
            raise ValueError(f"Missing required environment variable: {param}")
        params[param] = value
    
    return params

In [43]:
min_date = pd.Timestamp.min.replace(tzinfo=None)
max_date = pd.Timestamp.max.replace(tzinfo=None)

def generate_unique_id(val1, val2, val3):
    combined_str = f"{val1}{val2}{val3}"
    hash_object = hashlib.sha256(combined_str.encode())
    hash_bytes = hash_object.hexdigest()
    base64_hash = int(hash_bytes, 16)
    base64_hash = base64_hash % (10**9)
    return base64_hash

    

def safe_convert_to_datetime(x):
    try:
        timestamp = pd.Timestamp(x).tz_localize(None)
        if min_date <= timestamp <= max_date:
            return timestamp.strftime('%Y-%m-%d %H:%M:%S')
        else:
            return pd.NaT
    except (ValueError, TypeError):
        return pd.NaT



def format_timestamp(df):

    pattern = r'\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}\+\d{2}:\d{2}'
    column_list= ['FECHA','FECHA2']
    matching_columns = [col for col in df.columns if df[col].astype(str).str.contains(pattern).any()]
    for column_name in df.columns:
        if column_name in matching_columns:
            df[column_name] = df[column_name].apply(lambda x: safe_convert_to_datetime(x) if pd.notna(x) else x)
            df[column_name] = pd.to_datetime(df[column_name], errors='coerce')

            
    for column_name in column_list:
        if column_name in df:
            df[column_name] = df[column_name].apply(lambda x: pd.Timestamp(x).strftime('%Y-%m-%d %H:%M:%S') if pd.notna(x) else x)
            df[column_name] = df[column_name].astype('datetime64[ns]')

    df_end = df.replace(r'^\s*$', np.nan, regex=True) 
    return df_end

def infer_column_type(col, threshold=0.95):
    str_col = col.dropna().astype(str)
    total = len(str_col)
    if total == 0:
        return 'object'

    int_match = str_col.str.match(r'^-?\d+$')
    float_match = str_col.str.match(r'^-?(\d+)?\.\d+$')
    date_match = str_col.str.match(r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$')

    numeric_match = int_match | float_match

    ratio_int = int_match.sum() / total
    ratio_float = float_match.sum() / total
    ratio_numeric = numeric_match.sum() / total
    ratio_date = date_match.sum() / total

    if ratio_numeric >= threshold:
        return 'float'  
    elif ratio_date >= threshold:
        return 'date'
    else:
        return 'object'


def format_type_data(df):
    df1 = df.replace({np.nan: None})
    df2 = df1.replace({pd.NaT: None})
    df2.columns = [col.replace('.', '_').lower() for col in df2.columns]

    

    for column_name in df2.columns:
        inferred_type = infer_column_type(df2[column_name])

        if inferred_type == 'int':
            try:
                df2[column_name] = df2[column_name].replace({None: np.nan})  
                df2[column_name] = pd.to_numeric(df2[column_name], errors='coerce').fillna(-1).astype('int32')
            except Exception as e:
                print(f"Error converting {column_name} to int32: {e}")
                df2[column_name] = df2[column_name].astype('object')
    
        elif inferred_type == 'float':
            try:
                df2[column_name] = df2[column_name].replace({None: np.nan})  
                df2[column_name] = pd.to_numeric(df2[column_name], errors='coerce')
            except Exception as e:
                print(f"Error converting {column_name} to float: {e}")
                df2[column_name] = df2[column_name].astype('object')

        elif inferred_type == 'date':
            try:
                default_timestamp = pd.Timestamp('2000-01-01 00:00:00')
                df2[column_name] = pd.to_datetime(df2[column_name], errors='coerce').fillna(default_timestamp)
            except Exception as e:
                print(f"Error converting {column_name} to datetime: {e}")
                df2[column_name] = df2[column_name].astype('object')

        else:
            df2[column_name] = df2[column_name].astype('object')

    df3 = df2.replace([pd.NaT, '', np.nan, 'NaT'], None)
    return df3

def drop_table_if_exists():
    db_params = get_db_connection_params()
    conn = psycopg2.connect(
        dbname=db_params['DBNAME'],
        user=db_params['USER'],
        password=db_params['DB_PASSWORD'],
        host=db_params['ENDPOINT'],
        port=db_params['PORT']
    )
    conn.autocommit = True
    cursor = conn.cursor()
    
    try:
        logger.info("Dropping table public.source_support if it exists...")
        cursor.execute("DROP TABLE IF EXISTS user_01.source_support")
        logger.info("Table dropped.")
    except Exception as e:
        logger.error(f"Error dropping table: {str(e)}")
        raise
    finally:
        cursor.close()
        conn.close()

def pandas_dtype_to_spark(dtype):
    if np.issubdtype(dtype, np.integer):
        return IntegerType()
    elif np.issubdtype(dtype, np.floating):
        return DoubleType()
    elif np.issubdtype(dtype, np.datetime64):
        return TimestampType()
    else:
        return StringType()

In [51]:
def process_csv(input_path, output_path):

    try:
        
        df_1 = pd.read_csv(input_path, header=0, sep=',')
        df_2= format_timestamp(df_1)
        df_3= format_type_data(df_2)
        


        for col, dtype in {'diam_item': 'int32', 'peso_kg': 'float'}.items():
            df_3[col] = df_3[col].replace({None: np.nan})
            df_3[col] = pd.to_numeric(df_3[col], errors='coerce')
        
            if dtype.startswith('int'):
                df_3[col] = df_3[col].fillna(-1).astype(dtype)
            else:
                df_3[col] = df_3[col].astype(dtype)
        
        

                
        schema = StructType([
            StructField(col, pandas_dtype_to_spark(dtype), True)
            for col, dtype in zip(df_3.columns, df_3.dtypes)
        ])
        drop_table_if_exists()
        df_spark = sparkjum.createDataFrame(df_3, schema=schema)
        
        window_spec = Window.partitionBy(
            "e3did", "supportid", "supportmarkall", "support").orderBy(
            "e3did", "supportid", "supportmarkall", "support")
        
        df_spark = df_spark.withColumn("rw", row_number().over(window_spec).cast(IntegerType()))
        df_spark = df_spark.where(df_spark["e3did"].isNotNull())

        df_spark = df_spark.withColumn(
            "id_db", sha2(concat_ws("||", "e3did","supportid", "supportmarkall", "support"), 256)
        )

        
        logger.info(f" df_spark count rows: {df_spark.count()}")
        # df_spark.printSchema()

        
        db_params = get_db_connection_params()
        jdbc_url = f"jdbc:postgresql://{db_params['ENDPOINT']}:{db_params['PORT']}/{db_params['DBNAME']}"

        # Step 3: Write using JDBC
        df_spark.write \
            .format("jdbc") \
            .option("url", jdbc_url) \
            .option("dbtable", "user_01.source_support") \
            .option("user", db_params['USER']) \
            .option("password", db_params['DB_PASSWORD']) \
            .option("driver", "org.postgresql.Driver") \
            .option("batchsize", 10000) \
            .mode("append") \
            .save()

        logger.info("Data successfully written to Postgres via JDBC")
        
        return True
    except Exception as e:
        logger.error(f"Error processing CSV: {str(e)}")
        return False

In [52]:
def load_to_database_spark(input_path):
    conn = None
    cur = None
    
    try:
        # Step 1: Sample and infer types


        df_1 = pd.read_csv(input_path, header=0, sep=',')
        df_2= format_timestamp(df_1)
        df_3= df_2
        


        for col, dtype in {'diam_item': 'int32', 'peso_kg': 'float'}.items():
            df_3[col] = df_3[col].replace({None: np.nan})
            df_3[col] = pd.to_numeric(df_3[col], errors='coerce')
        
            if dtype.startswith('int'):
                df_3[col] = df_3[col].fillna(-1).astype(dtype)
            else:
                df_3[col] = df_3[col].astype(dtype)
        
        schema = StructType([
            StructField(col, pandas_dtype_to_spark(dtype), True)
            for col, dtype in zip(df_3.columns, df_3.dtypes)
        ])
        drop_table_if_exists()
        df_spark = sparkjum.createDataFrame(df_3, schema=schema)        

        window_spec = Window.partitionBy(
            "e3did", "supportid", "supportmarkall", "support").orderBy(
            "e3did", "supportid", "supportmarkall", "support")
        df_spark =df_spark.withColumn("RW", row_number().over(window_spec))

        df_spark = df_spark.withColumn("ID_DB",
                conv(sha2(concat_ws("||", col("e3did"), col("supportid"), col("supportmarkall")), 256), 16, 10).substr(1, 16)
                                          )


        logger.info(f"Columns after casting: {df_spark.dtypes}")

        with tempfile.TemporaryDirectory() as tmp_dir:
            output_path = os.path.join(tmp_dir, "spark_output")

            for c, dtype in df_spark.dtypes:
                if dtype == 'string':
                    df_spark = df_spark.withColumn(
                        c, 
                        regexp_replace(trim(col(c)), r'["\n\r]', ' ')
                    )

            df_spark.coalesce(1).write.csv(output_path, header=True, mode='overwrite')
            
            part_file = next(
                (f for f in os.listdir(output_path) 
                 if f.startswith("part-") and f.endswith(".csv")),
                None
            )
            
            if not part_file:
                raise FileNotFoundError("CSV part file not found")

            csv_file_path = os.path.join(output_path, part_file)

            db_params = get_db_connection_params()
            conn = psycopg2.connect(
                host=db_params['ENDPOINT'],
                port=db_params['PORT'],
                database=db_params['DBNAME'],
                user=db_params['USER'],
                password=db_params['DB_PASSWORD'],
                sslmode='require'
            )
            cur = conn.cursor()

            cur.execute("""
                SELECT EXISTS (
                    SELECT FROM information_schema.tables 
                    WHERE table_schema = 'public'
                    AND table_name = 'source_support'
                );
            """)
            
            if not cur.fetchone()[0]:
                columns_def = ', '.join(f"{col} TEXT" for col in df_spark.columns)
                create_table_query = f"""
                    CREATE TABLE public.source_support (
                        id SERIAL PRIMARY KEY,
                        {columns_def}
                    )
                """
                cur.execute(create_table_query)
                conn.commit()

            with open(csv_file_path, 'r', encoding='utf-8') as f:
                logger.info("Loading data into source_support...")
                columns = ', '.join(df_spark.columns)
                copy_sql = f"""
                    COPY public.source_support({columns})
                    FROM STDIN WITH CSV HEADER
                """
                cur.copy_expert(copy_sql, f)

            conn.commit()
            logger.info("Data successfully loaded")

    except Exception as e:
        if conn:
            conn.rollback()
        logger.error(f"Database operation failed: {str(e)}")
        raise

    finally:
        if cur:
            cur.close()
        if conn:
            conn.close()

In [53]:
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
process_csv(input_path,output_path)

[2025-04-24 22:22:09,306] INFO @ line 123: Dropping table public.source_support if it exists...
[2025-04-24 22:22:09,411] INFO @ line 125: Table dropped.
[2025-04-24 22:22:11,810] INFO @ line 42:  df_spark count rows: 9447
[2025-04-24 22:22:19,048] INFO @ line 61: Data successfully written to Postgres via JDBC


True